### Mount Google Drive

To access files stored in your Google Drive, you first need to mount it. This will prompt you to authenticate your Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, you can access your files. Here's an example of how to load a CSV file named `my_data.csv` located in the root of your Google Drive into a pandas DataFrame.

In [ ]:
import pandas as pd

# Replace 'my_data.csv' with the actual path to your CSV file in Google Drive
# For example, if your file is in 'My Drive/data/my_data.csv', the path would be '/content/drive/My Drive/data/my_data.csv'
file_path = '/content/drive/My Drive/Heart_Disease.csv'

try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the file path and ensure Google Drive is mounted correctly.")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

/tmp/ipykernel_8265/2128095330.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Data loaded successfully:


,Year,LocationAbbr,LocationDesc,GeographicLevel,DataSource,Class,Topic,Data_Value,Data_Value_Unit,Data_Value_Type,...,Data_Value_Footnote,Confidence_limit_Low,Confidence_limit_High,StratificationCategory1,Stratification1,StratificationCategory2,Stratification2,StratificationCategory3,Stratification3,LocationID
0,1999,AL,Autauga,County,NVSS,Cardiovascular Diseases,All heart disease,NaN,"per 100,000","Age-Standardized, Spatiotemporally Smoothed Rate",...,Value suppressed,NaN,NaN,Age group,Ages 35-64 years,Race,American Indian/Alaska Native,Sex,Overall,1001
1,2013,AL,Autauga,County,NVSS,Cardiovascular Diseases,All heart disease,NaN,"per 100,000","Age-Standardized, Spatiotemporally Smoothed Rate",...,Value suppressed,NaN,NaN,Age group,Ages 35-64 years,Race,American Indian/Alaska Native,Sex,Overall,1001
2,2014,AL,Autauga,County,NVSS,Cardiovascular Diseases,All heart disease,NaN,"per 100,000","Age-Standardized, Spatiotemporally Smoothed Rate",...,Value suppressed,NaN,NaN,Age group,Ages 35-64 years,Race,American Indian/Alaska Native,Sex,Overall,1001
3,2005,AL,Autauga,County,NVSS,Cardiovascular Diseases,All heart disease,NaN,"per 100,000","Age-Standardized, Spatiotemporally Smoothed Rate",...,Value suppressed,NaN,NaN,Age group,Ages 35-64 years,Race,American Indian/Alaska Native,Sex,Overall,1001
4,2012,AL,Autauga,County,NVSS,Cardiovascular Diseases,All heart disease,NaN,"per 100,000","Age-Standardized, Spatiotemporally Smoothed Rate",...,Value suppressed,NaN,NaN,Age group,Ages 35-64 years,Race,American Indian/Alaska Native,Sex,Overall,1001


In [ ]:
# ── CELL 2: Dtypes + null counts ─────────────────────────────────────────
print(df.dtypes.to_string())
print("\nNull counts:")
print(df.isnull().sum().to_string())

Year                           object
LocationAbbr                   object
LocationDesc                   object
GeographicLevel                object
DataSource                     object
Class                          object
Topic                          object
Data_Value                    float64
Data_Value_Unit                object
Data_Value_Type                object
Data_Value_Footnote_Symbol     object
Data_Value_Footnote            object
Confidence_limit_Low          float64
Confidence_limit_High         float64
StratificationCategory1        object
Stratification1                object
StratificationCategory2        object
Stratification2                object
StratificationCategory3        object
Stratification3                object
LocationID                      int64

Null counts:
Year                                0
LocationAbbr                        0
LocationDesc                        0
GeographicLevel                     0
DataSource                          

In [ ]:
# ── CELL 3: Key column unique values ─────────────────────────────────────
cols_to_check = ['GeographicLevel', 'Stratification1', 'StratificationCategory1',
                 'Topic', 'Category', 'Data_Value_Type', 'LocationDesc']
for col in cols_to_check:
    if col in df.columns:
        vals = df[col].unique()
        print(f"\n{col} ({len(vals)} unique):", vals[:15])
    else:
        print(f"\n{col}: *** COLUMN NOT FOUND ***")


GeographicLevel (1 unique): ['County']

Stratification1 (2 unique): ['Ages 35-64 years' 'Ages 65 years and older']

StratificationCategory1 (1 unique): ['Age group']

Topic (5 unique): ['All heart disease' 'All stroke' 'Coronary heart disease (CHD)'
 'Cardiovascular disease (CVD)' 'Heart failure']

Category: *** COLUMN NOT FOUND ***

Data_Value_Type (2 unique): ['Age-Standardized, Spatiotemporally Smoothed Rate' 'Total percent change']

LocationDesc (1829 unique): ['Autauga' 'Baldwin' 'Barbour' 'Bibb' 'Blount' 'Bullock' 'Butler'
 'Calhoun' 'Chambers' 'Cherokee' 'Choctaw' 'Clarke' 'Chilton' 'Clay'
 'Coffee']


In [ ]:
print("Shape:", df.shape)
print("\nAll columns:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. {col}")

Shape: (5770240, 21)

All columns:
   0. Year
   1. LocationAbbr
   2. LocationDesc
   3. GeographicLevel
   4. DataSource
   5. Class
   6. Topic
   7. Data_Value
   8. Data_Value_Unit
   9. Data_Value_Type
  10. Data_Value_Footnote_Symbol
  11. Data_Value_Footnote
  12. Confidence_limit_Low
  13. Confidence_limit_High
  14. StratificationCategory1
  15. Stratification1
  16. StratificationCategory2
  17. Stratification2
  18. StratificationCategory3
  19. Stratification3
  20. LocationID


In [ ]:
# ── CELL 4 FIXED: Geographic coverage ────────────────────────────────────
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

print("Year range:", df['Year'].min(), "to", df['Year'].max())
print("Unique years:", sorted(df['Year'].dropna().unique().astype(int).tolist()))
print("\nUnique LocationDesc count:", df['LocationDesc'].nunique())
print("Sample LocationDesc:", df['LocationDesc'].unique()[:10])
print("\nUnique LocationAbbr:", df['LocationAbbr'].unique()[:10])
print("\nLocationID range:", df['LocationID'].min(), "to", df['LocationID'].max())

geo_cols = [c for c in df.columns if any(k in c.lower() for k in ['lat','lon','geo','fips'])]
print("\nGeospatial-looking columns:", geo_cols)

Year range: 1999.0 to 2019.0
Unique years: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

Unique LocationDesc count: 1829
Sample LocationDesc: ['Autauga' 'Baldwin' 'Barbour' 'Bibb' 'Blount' 'Bullock' 'Butler'
 'Calhoun' 'Chambers' 'Cherokee']

Unique LocationAbbr: ['AL' 'AK' 'AZ' 'AR' 'CA' 'CO' 'CT' 'DE' 'FL' 'DC']

LocationID range: 1001 to 56045

Geospatial-looking columns: ['GeographicLevel']


In [ ]:
# ── CELL 5 FIXED: Value breakdown ────────────────────────────────────────
df['Data_Value'] = pd.to_numeric(df['Data_Value'], errors='coerce')

print("=== Data_Value_Type breakdown ===")
print(df['Data_Value_Type'].value_counts())

print("\n=== Topic × Data_Value_Type counts ===")
print(df.groupby(['Topic','Data_Value_Type']).size().to_string())

print("\n=== Stratification1 × StratificationCategory1 ===")
print(df.groupby(['StratificationCategory1','Stratification1']).size().to_string())

print("\n=== Stratification2 sample ===")
print(df['StratificationCategory2'].unique())
print(df['Stratification2'].unique()[:10])

print("\n=== Stratification3 sample ===")
print(df['StratificationCategory3'].unique())
print(df['Stratification3'].unique()[:10])

=== Data_Value_Type breakdown ===
Data_Value_Type
Age-Standardized, Spatiotemporally Smoothed Rate    5268480
Total percent change                                 501760
Name: count, dtype: int64

=== Topic × Data_Value_Type counts ===
Topic                         Data_Value_Type                                 
All heart disease             Age-Standardized, Spatiotemporally Smoothed Rate    1053696
                              Total percent change                                 100352
All stroke                    Age-Standardized, Spatiotemporally Smoothed Rate    1053696
                              Total percent change                                 100352
Cardiovascular disease (CVD)  Age-Standardized, Spatiotemporally Smoothed Rate    1053696
                              Total percent change                                 100352
Coronary heart disease (CHD)  Age-Standardized, Spatiotemporally Smoothed Rate    1053696
                              Total percent change     

In [ ]:
# ── CELL 6: Full sample rows ──────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 35)
print(df.sample(5, random_state=42).to_string())

           Year LocationAbbr LocationDesc GeographicLevel DataSource                    Class                         Topic  Data_Value Data_Value_Unit                                   Data_Value_Type Data_Value_Footnote_Symbol Data_Value_Footnote  Confidence_limit_Low  Confidence_limit_High StratificationCategory1          Stratification1 StratificationCategory2 Stratification2 StratificationCategory3 Stratification3  LocationID
802156   2016.0           GA      Madison          County       NVSS  Cardiovascular Diseases  Coronary heart disease (CHD)        72.2     per 100,000  Age-Standardized, Spatiotemporally Smoothed Rate                        NaN                 NaN                  58.7                   94.5               Age group         Ages 35-64 years                    Race           White                     Sex         Overall       13195
4065726  2004.0           SD      Yankton          County       NVSS  Cardiovascular Diseases             All heart disease       

In [ ]:
# ── CELL 7: County-level completeness ────────────────────────────────────
print("Total rows:", len(df))
print("Rows with non-null Data_Value:", df['Data_Value'].notna().sum())
print("Null % in Data_Value:", round(df['Data_Value'].isna().mean()*100, 1), "%")

print("\n=== Data_Value stats by Topic (Age-Standardized only) ===")
mask_smooth = df['Data_Value_Type'] == 'Age-Standardized, Spatiotemporally Smoothed Rate'
print(df[mask_smooth].groupby('Topic')['Data_Value'].describe().round(2))

print("\n=== Rows per year (Age-Standardized, All heart disease, Ages 35-64) ===")
sub = df[(df['Data_Value_Type']=='Age-Standardized, Spatiotemporally Smoothed Rate') &
         (df['Topic']=='All heart disease') &
         (df['Stratification1']=='Ages 35-64 years')]
print(sub.groupby('Year').size())
print("Counties per year sample:", sub.groupby('Year')['LocationDesc'].nunique().head())

Total rows: 5770240
Rows with non-null Data_Value: 3404765
Null % in Data_Value: 41.0 %

=== Data_Value stats by Topic (Age-Standardized only) ===
                                 count    mean     std   min    25%    50%  \
Topic                                                                        
All heart disease             621744.0  696.12  689.01  12.1  100.4  222.8   
All stroke                    621744.0  166.43  174.52   0.1   15.4   37.0   
Cardiovascular disease (CVD)  621744.0  928.44  917.80   3.9  124.3  275.4   
Coronary heart disease (CHD)  621744.0  436.86  460.55   1.9   64.6  148.1   
Heart failure                 621744.0  235.80  259.16   0.0   12.5   35.1   

                                 75%     max  
Topic                                         
All heart disease             1285.6  3993.4  
All stroke                     307.7  1521.0  
Cardiovascular disease (CVD)  1735.3  4188.4  
Coronary heart disease (CHD)   763.5  3142.7  
Heart failure           

In [ ]:
# ── CELL 8: LocationID — is it FIPS? ─────────────────────────────────────
print("LocationID sample values:", df['LocationID'].unique()[:20])
print("LocationID dtype:", df['LocationID'].dtype)
print("5-digit FIPS check (county FIPS are 5 digits):")
print(df['LocationID'].astype(str).str.len().value_counts())

# Check if LocationAbbr is state abbreviation
print("\nLocationAbbr unique count:", df['LocationAbbr'].nunique())
print("LocationAbbr sample:", sorted(df['LocationAbbr'].unique())[:15])

LocationID sample values: [1001 1003 1005 1007 1009 1011 1013 1015 1017 1019 1023 1025 1021 1027
 1031 1029 1033 1035 1037 1041]
LocationID dtype: int64
5-digit FIPS check (county FIPS are 5 digits):
LocationID
5    5199840
4     570400
Name: count, dtype: int64

LocationAbbr unique count: 51
LocationAbbr sample: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL']


In [ ]:
# ── CELL 9: The clean working subset we'll actually use ───────────────────
# Defining the canonical filter for all A3 analysis
TOPIC    = 'All heart disease'
DV_TYPE  = 'Age-Standardized, Spatiotemporally Smoothed Rate'
AGE      = 'Ages 35-64 years'   # younger age group — more variation across counties
RACE     = 'Overall'             # Stratification2
SEX      = 'Overall'             # Stratification3

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Data_Value'] = pd.to_numeric(df['Data_Value'], errors='coerce')

sub = df[
    (df['Topic']          == TOPIC)   &
    (df['Data_Value_Type']== DV_TYPE) &
    (df['Stratification1']== AGE)     &
    (df['Stratification2']== RACE)    &
    (df['Stratification3']== SEX)
].copy()

print("Filtered subset shape:", sub.shape)
print("Null Data_Value in subset:", sub['Data_Value'].isna().sum(),
      f"({sub['Data_Value'].isna().mean()*100:.1f}%)")
print("Counties:", sub['LocationDesc'].nunique())
print("Years:", sorted(sub['Year'].dropna().unique().astype(int).tolist()))

# How many counties have complete data across all 21 years?
county_year_counts = sub.groupby('LocationID')['Year'].count()
print("\nCounty year-coverage distribution:")
print(county_year_counts.value_counts().sort_index().tail(10))
print(f"\nCounties with all 21 years of non-null data:",
      sub.dropna(subset=['Data_Value']).groupby('LocationID')['Year'].count().eq(21).sum())

Filtered subset shape: (65856, 21)
Null Data_Value in subset: 1119 (1.7%)
Counties: 1829
Years: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

County year-coverage distribution:
Year
21    3136
Name: count, dtype: int64

Counties with all 21 years of non-null data: 3082


In [ ]:
# ── CELL 10: State-level aggregation check ────────────────────────────────
# Can we aggregate counties → states cleanly?
state_year = sub.dropna(subset=['Data_Value']).groupby(
    ['LocationAbbr', 'Year'])['Data_Value'].agg(['mean','count','std']).reset_index()

print("States × Years available:", state_year.shape)
print("\nMin counties per state-year:", state_year['count'].min())
print("Max counties per state-year:", state_year['count'].max())
print("\nSample:")
print(state_year[state_year['LocationAbbr']=='AL'].to_string())

States × Years available: (1071, 5)

Min counties per state-year: 1
Max counties per state-year: 242

Sample:
   LocationAbbr    Year        mean  count        std
21           AL  1999.0  194.950746     67  43.486544
22           AL  2000.0  191.046269     67  42.379106
23           AL  2001.0  188.419403     67  41.283166
24           AL  2002.0  186.619403     67  41.697497
25           AL  2003.0  189.117910     67  46.508301
26           AL  2004.0  184.910448     67  45.764890
27           AL  2005.0  184.844776     67  45.563856
28           AL  2006.0  185.289552     67  47.789228
29           AL  2007.0  173.450746     67  44.920861
30           AL  2008.0  174.807463     67  43.912379
31           AL  2009.0  177.289552     67  42.231925
32           AL  2010.0  178.682090     67  45.368431
33           AL  2011.0  173.771642     67  43.927588
34           AL  2012.0  176.192537     67  44.365453
35           AL  2013.0  183.352239     67  47.494591
36           AL  2014.0  1

In [ ]:
# ── CELL 11: Value range sanity check ────────────────────────────────────
print("Data_Value stats for our working subset:")
print(sub['Data_Value'].describe().round(2))

print("\nTop 5 highest mortality counties (any year):")
print(sub.nlargest(5, 'Data_Value')[
    ['Year','LocationAbbr','LocationDesc','Data_Value']].to_string())

print("\nTop 5 lowest mortality counties (any year):")
print(sub.nsmallest(5, 'Data_Value')[
    ['Year','LocationAbbr','LocationDesc','Data_Value']].to_string())

print("\nNational mean per year:")
print(sub.dropna(subset=['Data_Value']).groupby('Year')['Data_Value'].mean().round(2).to_string())

Data_Value stats for our working subset:
count    64737.00
mean       119.84
std         45.67
min         24.80
25%         86.20
50%        111.40
75%        146.20
max        377.70
Name: Data_Value, dtype: float64

Top 5 highest mortality counties (any year):
           Year LocationAbbr LocationDesc  Data_Value
1893518  2016.0           LA     Franklin       377.7
1893526  2014.0           LA     Franklin       376.0
1893530  2015.0           LA     Franklin       374.4
109324   2013.0           AL       Wilcox       374.3
2387404  2000.0           MS    Humphreys       368.7

Top 5 lowest mortality counties (any year):
          Year LocationAbbr LocationDesc  Data_Value
431876  2013.0           CO        Eagle        24.8
482283  2013.0           CO       Pitkin        24.9
430204  2013.0           CO      Douglas        25.3
431884  2011.0           CO        Eagle        25.3
482291  2011.0           CO       Pitkin        25.5

National mean per year:
Year
1999.0    136.79
20